**📘 Explicación Detallada del Caso de Estudio: Data Leakage en el Titanic**

Este experimento se centra en demostrar, de manera transparente y reproducible, uno de los problemas más críticos en la construcción de modelos predictivos: el data leakage o fuga de datos. Este fenómeno ocurre cuando el modelo tiene acceso —de forma explícita o implícita— a información que no estaría disponible en un escenario real de predicción, lo que genera métricas artificialmente infladas y una falsa sensación de desempeño.

En este caso de estudio, utilizamos el dataset del Titanic, uno de los conjuntos de datos más conocidos en ciencia de datos, donde el objetivo es predecir si un pasajero sobrevivió o no.

**1. ¿Qué es exactamente el Data Leakage?**

La fuga de datos consiste en que el modelo aprende patrones basados en información que, en la práctica, no debería conocer durante el entrenamiento, ya sea porque:

proviene del futuro,

se deriva directamente del objetivo,

está calculada a partir de datos agregados donde se mezclan muestras de entrenamiento y prueba,

o incluye señales ocultas que correlacionan casi perfectamente con la etiqueta.

Cuando esto sucede, el modelo obtiene un desempeño sobresaliente en validación, pero fracasa en producción, donde dicha información no existe.

**2. Objetivo del Experimento**

Este notebook persigue tres objetivos principales:

Entrenar un modelo correctamente, sin fuga de datos, siguiendo buenas prácticas.

Introducir deliberadamente una variable con fuga, mostrando cómo las métricas se distorsionan por completo.

Comparar ambos escenarios, evidenciando la importancia de construir pipelines robustos y libres de contaminación.

**3. Construcción del Modelo SIN Fuga de Datos**

Primero realizamos un preprocesamiento limpio:

selección de columnas relevantes,

tratamiento de valores faltantes,

conversión de variables categóricas a numéricas (one-hot encoding),

separación estricta entre entrenamiento y prueba.

En este escenario, el modelo obtiene un desempeño razonable para un dataset ruidoso y realista, típicamente con un accuracy entre 0.75 y 0.82.
Esto refleja un aprendizaje genuino de patrones socioeconómicos y demográficos reales.

**4. Construcción del Modelo CON Fuga de Datos**

En la segunda parte, introducimos una columna construida artificialmente:

survived_leak = survived


Esta variable replica exactamente el objetivo que el modelo intenta predecir.
Aunque este caso es extremo y transparente, sirve para ilustrar en forma didáctica lo que ocurre cuando una feature contiene de manera directa —o velada— la respuesta correcta.

El modelo, al tener acceso a esta señal, produce un accuracy cercano al 100%, un resultado imposible en un entorno real.
La métrica es perfecta porque el modelo no está aprendiendo: solo está leyendo la solución.

**5. Reflexiones y Riesgos Asociados**

Este ejemplo se concibió con un caso de fuga obvio, pero en la práctica la fuga suele ser:

sutil, como escalado hecho antes de dividir en train/test;

temporal, cuando se usan variables del futuro para predecir el pasado;

estructural, como identificadores que contienen patrones del objetivo;

estadística, como target encoding mal aplicado.

En entornos reales, estas fugas son difíciles de detectar y muy costosas, ya que llevan a modelos que fallan en producción, erosionan la confianza y generan sesgos operativos.

**6. Importancia para la Ingeniería de IA y MLOps**

Gestionar y prevenir la fuga de datos es un pilar fundamental en:

Diseño de pipelines seguros,

Validación robusta,

Gobernanza del ciclo de vida del modelo,

Sistemas productivos supervisados continuamente.

La prevención del data leakage no es solo una buena práctica técnica; es un requisito para garantizar la integridad, reproductibilidad y confiabilidad de los sistemas de IA.

**7. Conclusión del Caso**

El experimento demuestra con evidencia cuantitativa que:

Un modelo correctamente entrenado ofrece métricas realistas.

Un modelo con fuga de datos obtiene métricas espectaculares pero falsas.

La diferencia entre ambos escenarios es enorme y resalta el valor de validar cuidadosamente cada paso del pipeline.

Este caso de estudio constituye una base sólida para comprender uno de los errores más peligrosos —y comunes— en ciencia de datos, y proporciona una demostración clara que puede utilizarse tanto en docencia como en auditoría técnica.

# Notebook completo sobre Data Leakage
Explicación detallada de cada línea de código.

## Descarga del dataset Titanic

In [1]:
!wget -q https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv -O titanic.csv  # Descarga del dataset


## Importación de librerías

In [2]:
import pandas as pd  # Manipulación de datos
import numpy as np  # Operaciones numéricas
from sklearn.model_selection import train_test_split  # División Train/Test
from sklearn.linear_model import LogisticRegression  # Modelo de clasificación
from sklearn.metrics import accuracy_score, classification_report  # Métricas de evaluación


## Carga del dataset y exploración inicial

In [3]:
df = pd.read_csv('titanic.csv')  # Carga del archivo CSV
df.head()  # Vista previa


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Selección y limpieza de columnas

In [5]:
columnas = ['Survived','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']  # Features útiles
df = df[columnas].dropna().copy()  # Eliminación de filas incompletas
df.shape  # Tamaño final

(712, 8)

## One-hot encoding para variables categóricas

In [7]:
df_proc = pd.get_dummies(df, columns=['Sex','Embarked'], drop_first=True)  # Codificación
df_proc.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,True,False,True
1,1,1,38.0,1,0,71.2833,False,False,False
2,1,3,26.0,0,0,7.9250,False,False,True
3,1,1,35.0,1,0,53.1000,False,False,True
4,0,3,35.0,0,0,8.0500,True,False,True


## Entrenamiento del modelo SIN fuga de datos

In [10]:
y = df_proc['Survived']  # Variable objetivo
X = df_proc.drop('Survived',axis=1)  # Features
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)  # División de datos
model_no_leak = LogisticRegression(max_iter=1000)  # Modelo
model_no_leak.fit(X_train,y_train)  # Entrenamiento
pred_no_leak = model_no_leak.predict(X_test)  # Predicción
print('Accuracy sin fuga:', accuracy_score(y_test,pred_no_leak))  # Exactitud
print(classification_report(y_test,pred_no_leak))  # Reporte

Accuracy sin fuga: 0.7972027972027972
              precision    recall  f1-score   support

           0       0.82      0.85      0.83        85
           1       0.76      0.72      0.74        58

    accuracy                           0.80       143
   macro avg       0.79      0.79      0.79       143
weighted avg       0.80      0.80      0.80       143



## Creación de una fuga de datos artificial

In [12]:
df_leak = df_proc.copy()  # Copia del dataset
df_leak['survived_leak'] = df_leak['Survived']  # Columna filtrada del objetivo
X_leak = df_leak.drop('Survived',axis=1)  # Conjunto contaminado
y_leak = df_leak['Survived']
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak,y_leak,test_size=0.2,random_state=42,stratify=y_leak)  # División
model_leak = LogisticRegression(max_iter=1000)  # Modelo
model_leak.fit(X_train_l,y_train_l)  # Entrenamiento con fuga
pred_leak = model_leak.predict(X_test_l)  # Predicción
print('Accuracy con fuga:', accuracy_score(y_test_l,pred_leak))  # Exactitud inflada
print(classification_report(y_test_l,pred_leak))  # Reporte

Accuracy con fuga: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        85
           1       1.00      1.00      1.00        58

    accuracy                           1.00       143
   macro avg       1.00      1.00      1.00       143
weighted avg       1.00      1.00      1.00       143



## Conclusiones
La fuga de datos puede inflar artificialmente las métricas y llevar a modelos no confiables.